<a href="https://colab.research.google.com/github/jonik2909/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [ ]:
# 1. Configuration and Tokenizer
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# 2. Quantization Setup (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # Automatically handles GPU/CPU placement
    # dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# 4. Inference Test (Before Fine-tuning)
# prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

# Prepare inputs and move to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

# Decode and print results
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# first_block = model.model.layers[0]
# print("first_block:", first_block)
# print(first_block.self_attn)
# print(model.config)

A tokenizer is a tool in natural language processing that separates a sentence into its individual words, phrases, or sentences. A natural language processing (NLP) system needs a tokenizer to work with text data.

Beyond this, a natural language processing (NLP) system also requires a preprocessor, which is responsible for transforming the tokenized text into a format that can be understood by the various NLP modules.


In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including fronzen 4-bit): {total_params:,}")

Total parameters (including fronzen 4-bit): 615,606,272
